# Módulo 12 · Práctica: tu propio tokenizador BPE

Vas a construir desde cero un **tokenizador BPE a nivel de byte**, el mismo tipo que usan GPT-2 y muchos LLM actuales, y a entrenarlo sobre el inicio de *El Quijote* (dominio público, embebido abajo: unos 2,5 KB). Después medirás cuánto comprime (bytes por token) y por qué el español, los números o el ruso cuestan distinto.

**Cómo trabajar:** ejecuta en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que va detrás.

| Parte | Qué haces |
| --- | --- |
| 1 | Texto y UTF-8: por qué la ñ ocupa 2 bytes |
| 2 | Contar pares y fusionar |
| 3 | Entrenar BPE |
| 4 | Codificar y decodificar |
| 5 | Medir la compresión |
| 6 | Efectos curiosos: idiomas, números, medios caracteres |
| 7 | Pretokenización (demo) |
| 8 | Retos |

## Parte 1 · Texto y UTF-8

Usamos el primer capítulo de *El Quijote* (1605), en dominio público. No hay descargas: el texto va en la celda.

In [ ]:
TEXTO = 'En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero, salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, algún palomino de añadidura los domingos, consumían las tres partes de su hacienda. El resto della concluían sayo de velarte, calzas de velludo para las fiestas, con sus pantuflos de lo mesmo, y los días de entresemana se honraba con su vellorí de lo más fino. Tenía en su casa una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte, y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera. Frisaba la edad de nuestro hidalgo con los cincuenta años; era de complexión recia, seco de carnes, enjuto de rostro, gran madrugador y amigo de la caza. Quieren decir que tenía el sobrenombre de Quijada, o Quesada, que en esto hay alguna diferencia en los autores que deste caso escriben; aunque por conjeturas verosímiles se deja entender que se llamaba Quijana. Pero esto importa poco a nuestro cuento; basta que en la narración dél no se salga un punto de la verdad.\nEs, pues, de saber que este sobredicho hidalgo, los ratos que estaba ocioso, que eran los más del año, se daba a leer libros de caballerías, con tanta afición y gusto, que olvidó casi de todo punto el ejercicio de la caza, y aun la administración de su hacienda. Y llegó a tanto su curiosidad y desatino en esto, que vendió muchas hanegas de tierra de sembradura para comprar libros de caballerías en que leer, y así, llevó a su casa todos cuantos pudo haber dellos; y de todos, ningunos le parecían tan bien como los que compuso el famoso Feliciano de Silva, porque la claridad de su prosa y aquellas entricadas razones suyas le parecían de perlas, y más cuando llegaba a leer aquellos requiebros y cartas de desafíos, donde en muchas partes hallaba escrito: «La razón de la sinrazón que a mi razón se hace, de tal manera mi razón enflaquece, que con razón me quejo de la vuestra fermosura». Y también cuando leía: «Los altos cielos que de vuestra divinidad divinamente con las estrellas os fortifican, y os hacen merecedora del merecimiento que merece la vuestra grandeza».\nCon estas razones perdía el pobre caballero el juicio, y desvelábase por entenderlas y desentrañarles el sentido, que no se lo sacara ni las entendiera el mesmo Aristóteles, si resucitara para solo ello.'
print(len(TEXTO), 'caracteres')

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

for s in ["a", "ñ", "á", "año", "€", "😀"]:
    b = s.encode("utf-8")
    print(f"{s!r:6} {len(s)} caracteres -> {len(b)} bytes {list(b)}  {[format(x, '08b') for x in b]}")

Un carácter con código mayor que 127 usa el patrón `110xxxxx 10xxxxxx` (2 bytes). La ñ es U+00F1 = 241, y sus 11 bits se reparten así: `11000011 10110001` = C3 B1.

**Ejercicio 1.** Calcula `n_bytes` (bytes UTF-8 de `TEXTO`) y `extra`, la diferencia entre bytes y caracteres. Crea también `no_ascii`, un `Counter` con los caracteres de `TEXTO` cuyo código (`ord`) es mayor que 127.

In [ ]:
# TU CÓDIGO
n_bytes = None
extra = None
no_ascii = None

In [ ]:
# Comprobación
assert n_bytes == 2536, f"Esperaba 2536 bytes y tienes {n_bytes}"
assert extra == 52, "Cada carácter no ASCII de este texto añade 1 byte extra"
assert no_ascii["ñ"] == 4 and sum(no_ascii.values()) == 52, "Revisa el filtro ord(c) > 127"
print("Correcto:", dict(no_ascii))

## Parte 2 · Contar pares y fusionar

Trabajamos con listas de enteros: al principio, los bytes (0–255). Cada fusión crea un entero nuevo (256, 257, …).

**Ejercicio 2.** Escribe `contar_pares(ids)`, que devuelve un `Counter` con los pares adyacentes, y `fusionar(ids, par, nuevo)`, que sustituye cada aparición de `par` (de izquierda a derecha, sin solapes) por `nuevo`.

In [ ]:
def contar_pares(ids):
    # TU CÓDIGO
    pass

def fusionar(ids, par, nuevo):
    # TU CÓDIGO
    pass

In [ ]:
# Comprobación
assert contar_pares([1, 2, 3, 1, 2]) == Counter({(1, 2): 2, (2, 3): 1, (3, 1): 1}), "contar_pares falla"
assert fusionar([1, 2, 3, 1, 2], (1, 2), 99) == [99, 3, 99], "fusionar falla"
assert fusionar([5, 5, 5, 5], (5, 5), 7) == [7, 7], "Sin solapes: 'aaaa' -> 'aa aa'"
assert fusionar([5, 5, 5], (5, 5), 7) == [7, 5], "De izquierda a derecha"
print("Correcto")

## Parte 3 · Entrenar BPE

El algoritmo de la sección 12.2: contar, elegir el par más frecuente, fusionar y anotar, `n` veces.

**Ejercicio 3.** Completa `entrenar(texto, n)`. Devuelve `merges` (diccionario `par -> nuevo id`, en orden) y `vocab` (diccionario `id -> bytes`). Usa `max(c, key=c.get)` para elegir el par: en caso de empate gana el que apareció antes, y así todos obtenemos el mismo resultado.

In [ ]:
def entrenar(texto, n):
    ids = list(texto.encode("utf-8"))
    merges = {}
    vocab = {i: bytes([i]) for i in range(256)}
    for k in range(n):
        # TU CÓDIGO: contar, elegir par, fusionar, guardar en merges y vocab
        pass
    return merges, vocab

In [ ]:
# Comprobación 1: ejemplo clásico "aaabdaaabac"
m, v = entrenar("aaabdaaabac", 3)
assert [v[i] for i in m.values()] == [b"aa", b"aaa", b"aaab"], f"Fusiones: {[v[i] for i in m.values()]}"
# Comprobación 2: El Quijote
merges, vocab = entrenar(TEXTO, 300)
assert len(merges) == 300 and len(vocab) == 556, "Debe haber 256 + 300 tokens"
assert vocab[256] == b"e ", f"La primera fusión debería ser 'e ' y es {vocab[256]!r}"
print("Correcto. Primeras 15 fusiones:")
for t in range(256, 271):
    print(t, repr(vocab[t].decode("utf-8", errors="replace")))

Las primeras fusiones capturan terminaciones con espacio («e␣», «a␣», «s␣») y palabras frecuentes («de␣», «que␣»). Busca dónde aparecen los tokens de la ñ y las tildes:

In [ ]:
for t, b in vocab.items():
    if t >= 256 and b"\xc3" in b:
        print(t, "puesto", t - 255, repr(b.decode("utf-8", errors="replace")))

## Parte 4 · Codificar y decodificar

Para codificar un texto nuevo se aplican las fusiones **en el orden en que se aprendieron**. Truco: en cada paso, entre los pares presentes, fusiona el que tenga el id de fusión más pequeño.

**Ejercicio 4.** Completa `codificar(texto, merges)` y `decodificar(ids, vocab)` (une los bytes y decodifica con `errors="replace"`).

In [ ]:
def codificar(texto, merges):
    ids = list(texto.encode("utf-8"))
    while len(ids) >= 2:
        # TU CÓDIGO: elige el par con menor merges.get(par, inf); si no está en merges, para
        pass
    return ids

def decodificar(ids, vocab):
    # TU CÓDIGO
    pass

In [ ]:
# Comprobación
assert codificar("aaabdaaabac", m) == [258, 100, 258, 97, 99], "Codificación de 'aaabdaaabac' incorrecta"
ids = codificar(TEXTO, merges)
assert decodificar(ids, vocab) == TEXTO, "Codificar y decodificar debe devolver el texto exacto"
assert decodificar(codificar("¡Ñandú pingüino 😀!", merges), vocab) == "¡Ñandú pingüino 😀!", "Debe funcionar con cualquier texto"
assert abs(len(ids) - 819) <= 5, f"Esperaba unos 819 tokens y tienes {len(ids)}"
def ver(s):
    return [vocab[i].decode("utf-8", errors="replace") for i in codificar(s, merges)]
print(len(ids), "tokens")
for w in ["caballerías", "hidalgo", "razón", "rocín"]:
    print(w, ver(w))

## Parte 5 · Medir la compresión

$$\text{compresión} = \frac{\text{bytes}}{\text{tokens}}$$

**Ejercicio 5.** Para cada `N` de la lista, quédate con las `N` primeras fusiones (`dict(list(merges.items())[:N])`), codifica `TEXTO` y guarda en `bpt` los bytes por token.

In [ ]:
Ns = [0, 10, 25, 50, 100, 200, 300]
bpt = []
# TU CÓDIGO

In [ ]:
# Comprobación
assert len(bpt) == len(Ns) and bpt[0] == 1.0, "Con 0 fusiones, 1 byte por token"
assert all(a < b for a, b in zip(bpt, bpt[1:])), "La compresión debe crecer con las fusiones"
assert abs(bpt[-1] - 3.10) < 0.03, f"Con 300 fusiones esperaba ≈ 3,10 bytes/token y tienes {bpt[-1]:.3f}"
for N, x in zip(Ns, bpt):
    print(f"{N:4d} fusiones -> {x:.3f} bytes/token")
plt.plot(Ns, bpt, "o-"); plt.xlabel("fusiones"); plt.ylabel("bytes por token"); plt.title("Compresión del BPE"); plt.show()

Medir sobre el texto de entrenamiento es hacer trampa (módulo 8). **Ejercicio 6.** Entrena con los dos primeros párrafos (`tr`) y 200 fusiones, y calcula los bytes por token en `tr` (`bpt_tr`) y en el tercer párrafo (`bpt_te`).

In [ ]:
parrafos = TEXTO.split("\n")
tr, te = "\n".join(parrafos[:2]), parrafos[2]
# TU CÓDIGO
bpt_tr = None
bpt_te = None

In [ ]:
# Comprobación
assert bpt_tr is not None and bpt_te is not None, "Calcula ambas"
assert bpt_te < bpt_tr, "El texto no visto debería comprimirse peor"
print(f"entrenamiento {bpt_tr:.3f} · validación {bpt_te:.3f}")

## Parte 6 · Efectos curiosos

El mismo mensaje no cuesta lo mismo en todos los idiomas. Nuestro tokenizador solo ha visto español:

In [ ]:
frases = {"español": "El perro ladra en la plaza.",
          "inglés": "The dog barks in the square.",
          "ruso": "Собака лает.",
          "número": "1605"}
for idioma, f in frases.items():
    nb, nt = len(f.encode()), len(codificar(f, merges))
    print(f"{idioma:8} {nb:3d} bytes -> {nt:3d} tokens ({nb/nt:.2f} bytes/token)  {ver(f)}")

El ruso sale a 1 token por byte: ninguna fusión aprendida encaja con el cirílico. Los números quedan dígito a dígito: no hay fusiones de cifras.

**Ejercicio 7 (medios caracteres).** Busca en `vocab` los tokens cuyos bytes **no** se pueden decodificar como UTF-8 válido (usa `try: b.decode("utf-8")` / `except UnicodeDecodeError`). Guarda sus ids en `rotos`. ¿Qué tienen en común?

In [ ]:
rotos = []
# TU CÓDIGO

In [ ]:
# Comprobación
assert 195 in rotos, "El byte 0xC3 (195) solo no es UTF-8 válido"
assert all(t not in rotos for t in range(128)), "Los bytes ASCII siempre son válidos"
print(len(rotos), "tokens no decodificables solos; entre los aprendidos:",
      [vocab[t] for t in rotos if t >= 256])

## Parte 7 · Pretokenización (demo)

Sin límites entre palabras, BPE aprende tokens que cruzan frases. Mira los más largos:

In [ ]:
print(sorted((b.decode("utf-8", "replace") for b in vocab.values()), key=len)[-3:])

GPT-2 primero trocea con una expresión regular en «palabras con su espacio delante» y aplica BPE dentro de cada trozo. Aquí, una versión simplificada:

In [ ]:
import re
PAT = re.compile(r" ?\w+| ?[^\w\s]+|\s+")
trozos = PAT.findall(TEXTO)
print(len(trozos), trozos[:8])
assert "".join(trozos) == TEXTO

def entrenar_pre(texto, n):
    frec = Counter(PAT.findall(texto))
    palabras = {tuple(w.encode()): f for w, f in frec.items()}
    merges, vocab = {}, {i: bytes([i]) for i in range(256)}
    for k in range(n):
        c = Counter()
        for w, f in palabras.items():
            for p in zip(w, w[1:]):
                c[p] += f
        if not c or max(c.values()) < 2:
            break
        par = max(c, key=c.get); nuevo = 256 + k
        merges[par] = nuevo; vocab[nuevo] = vocab[par[0]] + vocab[par[1]]
        palabras = {tuple(fusionar(list(w), par, nuevo)): f for w, f in palabras.items()}
    return merges, vocab

mp, vp = entrenar_pre(TEXTO, 300)
ids_p = [i for t in PAT.findall(TEXTO) for i in codificar(t, mp)]
print(len(mp), "fusiones ·", round(n_bytes / len(ids_p), 3), "bytes/token")
print("Tokens más largos:", sorted((b.decode("utf-8", "replace") for b in vp.values()), key=len)[-5:])

Comprime algo menos sobre este texto, pero sus tokens son palabras reales (« caballerías», « hacienda») que generalizan a otros textos.

## Parte 8 · Retos

1. **Cuenta letras.** Escribe una función que diga cuántas «r» tiene cada token de `ver("ferrocarril")`. ¿Qué debe saber el modelo para contestar?
2. **Vocabulario y coste.** Si cada token tiene un embedding de d = 4.096, ¿cuántos parámetros añade cada fusión? ¿Y 100.000 fusiones?
3. **Unigram.** Programa con programación dinámica la segmentación más probable de «casas» con el vocabulario de la sección 12.4 y comprueba que es «casa · s».
4. **Tu texto.** Entrena con otro texto tuyo (mensajes, apuntes) y compara los bytes por token.

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# --- Ejercicio 1 ---
# n_bytes = len(TEXTO.encode("utf-8"))
# extra = n_bytes - len(TEXTO)
# no_ascii = Counter(c for c in TEXTO if ord(c) > 127)
# --- Ejercicio 2 ---
# def contar_pares(ids):
#     return Counter(zip(ids, ids[1:]))
#
# def fusionar(ids, par, nuevo):
#     out, i = [], 0
#     while i < len(ids):
#         if i < len(ids) - 1 and (ids[i], ids[i + 1]) == par:
#             out.append(nuevo); i += 2
#         else:
#             out.append(ids[i]); i += 1
#     return out
# --- Ejercicio 3 ---
# def entrenar(texto, n):
#     ids = list(texto.encode("utf-8"))
#     merges = {}
#     vocab = {i: bytes([i]) for i in range(256)}
#     for k in range(n):
#         c = contar_pares(ids)
#         if not c:
#             break
#         par = max(c, key=c.get)
#         nuevo = 256 + k
#         ids = fusionar(ids, par, nuevo)
#         merges[par] = nuevo
#         vocab[nuevo] = vocab[par[0]] + vocab[par[1]]
#     return merges, vocab
# --- Ejercicio 4 ---
# def codificar(texto, merges):
#     ids = list(texto.encode("utf-8"))
#     while len(ids) >= 2:
#         c = contar_pares(ids)
#         par = min(c, key=lambda p: merges.get(p, float("inf")))
#         if par not in merges:
#             break
#         ids = fusionar(ids, par, merges[par])
#     return ids
#
# def decodificar(ids, vocab):
#     return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")
# --- Ejercicio 5 ---
# Ns = [0, 10, 25, 50, 100, 200, 300]
# bpt = []
# for N in Ns:
#     mN = dict(list(merges.items())[:N])
#     bpt.append(n_bytes / len(codificar(TEXTO, mN)))
# --- Ejercicio 6 ---
# parrafos = TEXTO.split("\n")
# tr, te = "\n".join(parrafos[:2]), parrafos[2]
# m2, v2 = entrenar(tr, 200)
# bpt_tr = len(tr.encode()) / len(codificar(tr, m2))
# bpt_te = len(te.encode()) / len(codificar(te, m2))
# --- Ejercicio 7 ---
# rotos = []
# for t, b in vocab.items():
#     try:
#         b.decode("utf-8")
#     except UnicodeDecodeError:
#         rotos.append(t)